# 10. Clustering de Estados por Perfil de Mortalidad

El estudio de caso de West Virginia plantea una pregunta de alcance más amplio: ¿Es este estado un caso aislado, o pertenece a un grupo de estados con un perfil de mortalidad estructuralmente similar?

Para responder esta pregunta se aplica un análisis de conglomerados K-Means sobre los 50 estados más el Distrito de Columbia, usando como variables las tasas ajustadas de mortalidad por causa en **2017**.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
# Preparación
datos_2017 = estados[(estados['year']==2017)&(estados['cause_name']!='All causes')&
                      estados['age_adjusted_death_rate'].notna()]
matriz = datos_2017.pivot_table(index='state', columns='cause_name',
                                 values='age_adjusted_death_rate', fill_value=0)
print(f"Dimensiones: {matriz.shape}")

# Estandarización
sc = StandardScaler()
mat_sc = sc.fit_transform(matriz)

# Método del codo
inercias = [KMeans(n_clusters=k, random_state=42, n_init=25).fit(mat_sc).inertia_ for k in range(1,11)]

fig_codo = go.Figure()
fig_codo.add_trace(go.Scatter(x=list(range(1,11)), y=inercias, mode='lines+markers',
                               line=dict(color='#1D3557', width=2),
                               marker=dict(color='#E63946', size=8)))
fig_codo.add_vline(x=4, line_dash='dash', line_color='#457B9D',
                   annotation_text='k = 4 (punto de codo)')
fig_codo.update_layout(title='Método del codo – número óptimo de clústeres',
                        xaxis_title='Número de clústeres (k)',
                        yaxis_title='Inercia total (within-cluster SS)',
                        height=380, template='plotly_white')
fig_codo.show()

Dimensiones: (51, 10)


In [3]:
# K-Means con k=4
km = KMeans(n_clusters=4, random_state=42, n_init=25)
clusters = km.fit_predict(mat_sc)
resultado = pd.DataFrame({'state':matriz.index, 'cluster':clusters+1}).sort_values(['cluster','state'])

print("Composición de los 4 clústeres:")
for c in sorted(resultado['cluster'].unique()):
    states = resultado[resultado['cluster']==c]['state'].tolist()
    print(f"\nClúster {c} ({len(states)} estados):")
    print("  " + ", ".join(states))

Composición de los 4 clústeres:

Clúster 1 (19 estados):
  Alaska, Arizona, Colorado, Idaho, Iowa, Minnesota, Montana, Nebraska, Nevada, New Hampshire, New Mexico, North Dakota, Oregon, South Dakota, Utah, Vermont, Washington, Wisconsin, Wyoming

Clúster 2 (12 estados):
  California, Connecticut, District of Columbia, Florida, Hawaii, Illinois, Maryland, Massachusetts, New Jersey, New York, Rhode Island, Virginia

Clúster 3 (12 estados):
  Delaware, Georgia, Indiana, Kansas, Maine, Michigan, Missouri, North Carolina, Ohio, Pennsylvania, South Carolina, Texas

Clúster 4 (8 estados):
  Alabama, Arkansas, Kentucky, Louisiana, Mississippi, Oklahoma, Tennessee, West Virginia


In [4]:
# Perfil de tasas por clúster
causas_perfil = ['Heart disease','Cancer','Unintentional injuries',"Alzheimer's disease",'CLRD','Stroke']
datos_perfil = (datos_2017[datos_2017['cause_name'].isin(causas_perfil)]
                .merge(resultado, on='state')
                .groupby(['cluster','cause_name'])['age_adjusted_death_rate'].mean().reset_index())

fig_perf = px.bar(datos_perfil, x='cause_name', y='age_adjusted_death_rate', color='cluster',
                   barmode='group', title='Tasa ajustada promedio por causa y clúster (2017)',
                   labels={'age_adjusted_death_rate':'Tasa por 100,000','cause_name':''},
                   color_discrete_map={1:'#A8DADC',2:'#457B9D',3:'#E63946',4:'#1D3557'},
                   template='plotly_white', text_auto='.1f')
fig_perf.update_layout(height=430, xaxis_tickangle=30,
                        legend=dict(orientation='h', y=-0.2))
fig_perf.show()

In [5]:
# PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(mat_sc)
df_pca = pd.DataFrame({'PC1':coords[:,0],'PC2':coords[:,1],
                        'cluster':clusters.astype(str), 'estado':matriz.index})

cluster_labels = {"0":"Clúster 1 – Bajo riesgo","1":"Clúster 2 – Riesgo medio-bajo",
                   "2":"Clúster 3 – ALTO RIESGO (WV)","3":"Clúster 4 – Riesgo medio"}
df_pca['cluster_label'] = df_pca['cluster'].map(cluster_labels)

fig_pca = px.scatter(df_pca, x='PC1', y='PC2', color='cluster_label', text='estado',
                      title='Visualización PCA – Clustering de estados (2017)',
                      template='plotly_white',
                      color_discrete_sequence=['#A8DADC','#457B9D','#E63946','#1D3557'])
fig_pca.update_traces(textposition='top center', marker=dict(size=9))
fig_pca.update_layout(height=520, legend=dict(orientation='h', y=-0.15))
fig_pca.show()

**Interpretación:** El clúster de alto riesgo agrupa a West Virginia junto con otros estados del sur y los Apalaches con tasas elevadas en casi todas las causas. El clúster de bajo riesgo concentra estados del noreste y algunos del oeste con mejores indicadores de salud pública. La agrupación confirma que West Virginia no es un caso aislado sino el extremo de un patrón estructural persistente.